In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

from utils import forward_selection, mdc_select, bcorsis_select, kfilter_select

In [ ]:
housing = fetch_california_housing()
X_real = housing.data
y      = housing.target

feature_names_real = list(housing.feature_names)
n, p_real = X_real.shape
print(f"California Housing: {n} samples, {p_real} real features")

rng_noise = np.random.default_rng(42)

n_copies  = 7
copy_src  = rng_noise.choice(p_real, size=n_copies, replace=False)
X_copies  = X_real[:, copy_src] + 0.1 * rng_noise.standard_normal((n, n_copies))
copy_names = [f"NoisyCopy_{feature_names_real[j]}" for j in copy_src]

n_lincomb  = 7
W          = rng_noise.standard_normal((p_real, n_lincomb))
W         /= np.linalg.norm(W, axis=0, keepdims=True)
X_lincomb  = X_real @ W + 0.1 * rng_noise.standard_normal((n, n_lincomb))
lincomb_names = [f"LinCombo_{i+1}" for i in range(n_lincomb)]

n_pure   = 500
X_pure   = rng_noise.standard_normal((n, n_pure))
pure_names = [f"PureNoise_{i+1}" for i in range(n_pure)]

X_noise     = np.hstack([X_copies, X_lincomb, X_pure])
noise_names = copy_names + lincomb_names + pure_names
p_noise     = X_noise.shape[1]

X_all      = np.hstack([X_real, X_noise])
feat_names = feature_names_real + noise_names
p_total    = X_all.shape[1]

scaler = StandardScaler()
X_all  = scaler.fit_transform(X_all)

print(f"Noise features: {n_copies} noisy copies + {n_lincomb} linear combos + {n_pure} pure noise = {p_noise}")
print(f"Total features: {p_total}")

In [ ]:
n_total = len(y)
half    = n_total // 2

rng_split = np.random.default_rng(0)
perm      = rng_split.permutation(n_total)

X_screen_all = X_all[perm[:half]]
y_screen_all = y[perm[:half]]

X_reg = X_all[perm[half:]]
y_reg = y[perm[half:]]

X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=0
)

print(f"Screening half : {len(y_screen_all)} samples")
print(f"Regression half: {len(y_reg)} samples  (Train: {len(y_train)} | Test: {len(y_test)})")

In [ ]:
N_SCREEN = 2000
K_NN     = 10

rng = np.random.default_rng(1)
if len(y_screen_all) > N_SCREEN:
    sc_idx = rng.choice(len(y_screen_all), size=N_SCREEN, replace=False)
    X_sc = X_screen_all[sc_idx]
    y_sc = y_screen_all[sc_idx]
else:
    X_sc = X_screen_all
    y_sc = y_screen_all

print("Running NNCMI (k=10) forward selection...", flush=True)
nncmi_feats_k10 = forward_selection(X_sc, y_sc, k_nn=10, statistic="nncmi")

print("Running NNCMI (k=25) forward selection...", flush=True)
nncmi_feats_k25 = forward_selection(X_sc, y_sc, k_nn=25, statistic="nncmi")

print("Running Chatterjee (kfoci) forward selection...", flush=True)
chatt_feats = forward_selection(X_sc, y_sc, k_nn=K_NN, statistic="chatterjee")

print("Running MDC screening (own stopping)...", flush=True)
mdc_feats = mdc_select(X_sc, y_sc)

m = max(len(nncmi_feats_k10), len(nncmi_feats_k25), len(chatt_feats))
print(f"Running MDC screening (matched m={m})...", flush=True)
mdc_feats_matched = mdc_select(X_sc, y_sc, nsis=m)

print("Running BcorSIS screening...", flush=True)
bcorsis_feats = bcorsis_select(X_sc, y_sc, nsis=m)

print("Running Kfilter screening...", flush=True)
kfilter_feats = kfilter_select(X_sc, y_sc, nsis=m)

print("Done.", flush=True)

In [ ]:
def noise_type(idx, p_real, n_copies, n_lincomb):
    if idx < p_real:
        return "Real"
    offset = idx - p_real
    if offset < n_copies:
        return "Noise:Copy"
    elif offset < n_copies + n_lincomb:
        return "Noise:LinCombo"
    else:
        return "Noise:Pure"


def summarise_selection(label, indices, feat_names, p_real, n_copies, n_lincomb):
    rows = []
    for rank, idx in enumerate(indices, start=1):
        rows.append({
            "Rank":    rank,
            "Index":   idx,
            "Feature": feat_names[idx],
            "Type":    noise_type(idx, p_real, n_copies, n_lincomb),
        })
    print(f"\n{label} — {len(indices)} features selected:")
    display(pd.DataFrame(rows))


summarise_selection("NNCMI (k=10)",  nncmi_feats_k10, feat_names, p_real, n_copies, n_lincomb)
summarise_selection("NNCMI (k=25)",  nncmi_feats_k25, feat_names, p_real, n_copies, n_lincomb)
summarise_selection("Chatterjee (kfoci)", chatt_feats, feat_names, p_real, n_copies, n_lincomb)
summarise_selection(
    f"MDC (own, nsis=⌊{N_SCREEN}/log({N_SCREEN})⌋={int(N_SCREEN/np.log(N_SCREEN))})",
    sorted(mdc_feats), feat_names, p_real, n_copies, n_lincomb
)
summarise_selection(f"MDC (matched, m={m})", sorted(mdc_feats_matched), feat_names, p_real, n_copies, n_lincomb)
summarise_selection("BcorSIS", sorted(bcorsis_feats), feat_names, p_real, n_copies, n_lincomb)
summarise_selection("Kfilter", sorted(kfilter_feats), feat_names, p_real, n_copies, n_lincomb)

In [ ]:
def fit_and_eval(X_train, X_test, y_train, y_test, feature_indices):
    model = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        random_state=0, n_jobs=-1, verbosity=0,
    )
    model.fit(
        X_train[:, feature_indices], y_train,
        eval_set=[(X_test[:, feature_indices], y_test)],
        verbose=False,
    )
    y_pred = model.predict(X_test[:, feature_indices])
    mse  = mean_squared_error(y_test, y_pred)
    return mse, np.sqrt(mse)


scenarios = {
    "NNCMI (k=10)":       nncmi_feats_k10,
    "NNCMI (k=25)":       nncmi_feats_k25,
    "Chatterjee (kfoci)": chatt_feats,
    "BcorSIS":            bcorsis_feats,
    "Kfilter":            kfilter_feats,
    "MDC (matched m)":    mdc_feats_matched,
    "MDC (own)":          mdc_feats,
    "All features":       list(range(p_total)),
    "Oracle (real only)": list(range(p_real)),
}

rows = []
for name, feats in scenarios.items():
    print(f"Fitting {name}...", flush=True)
    mse, rmse = fit_and_eval(X_train, X_test, y_train, y_test, feats)
    n_real_sel = sum(1 for f in feats if f < p_real)
    rows.append({
        "Method":           name,
        "# Selected":       len(feats),
        "# Real Selected":  n_real_sel,
        "# Noise Selected": len(feats) - n_real_sel,
        "Test MSE":         round(mse,  4),
        "Test RMSE":        round(rmse, 4),
    })

results_df = pd.DataFrame(rows).set_index("Method")
display(results_df)

In [ ]:
latex = results_df.to_latex(column_format="|l|c|c|c|c|c|", escape=False)
latex = (
    latex
    .replace("\\toprule",    "\\hline")
    .replace("\\midrule",    "\\hline")
    .replace("\\bottomrule", "\\hline")
)
print(latex)